<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/indicator/temp/impulse_work_219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyotp --q
!pip install smartapi-python==1.4.1 --q
!pip install logzero --q

In [ ]:

# package import statement
from SmartApi import SmartConnect #or from SmartApi.smartConnect import SmartConnect
import pyotp
from logzero import logger
import time
import os
import urllib
import json
import pandas as pd
import datetime as dt

api_key = 'xOHnB7MG'
username = 'M55123447'
pwd = '1471'
smartApi = SmartConnect(api_key)
try:
    token = "GJZACUQI2TTAIBHBA34XNFJURQ"
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e

correlation_id = "abcde"
data = smartApi.generateSession(username, pwd, totp)

if data['status'] == False:
    logger.error(data)

else:
    # login api call
    # logger.info(f"You Credentials: {data}")
    authToken = data['data']['jwtToken']
    refreshToken = data['data']['refreshToken']
    # fetch the feedtoken
    feedToken = smartApi.getfeedToken()
    # fetch User Profile
    res = smartApi.getProfile(refreshToken)
    smartApi.generateToken(refreshToken)
    res=res['data']['exchanges']
#Download Nifty50 Index Data
params = {
           "exchange": "NSE",
           "symboltoken": '99926000',
           "interval": "ONE_DAY",
           "fromdate": (dt.datetime(2020, 9, 21).strftime('%Y-%m-%d %H:%M')),
           "todate": (dt.datetime.today().strftime('%Y-%m-%d %H:%M'))
         }
nifty_data = smartApi.getCandleData(params)
nifty_data_format= pd.DataFrame(nifty_data["data"],
                               columns = ["Date","Open","High","Low","Close","Volume"])
nifty_data_format.set_index("Date",inplace=True)
nifty_data_format.index = pd.to_datetime(nifty_data_format.index)
nifty_data_format.index = nifty_data_format.index.tz_localize(None)

In [ ]:
df = nifty_data_format.copy()

# --------------------------
# Moving Average
# --------------------------
df["ma20"] = df["Close"].rolling(20).mean()

# --------------------------
# EMA calculation
# --------------------------
df["ema12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["ema26"] = df["Close"].ewm(span=26, adjust=False).mean()
df["ema13"] = df["Close"].ewm(span=13, adjust=False).mean()
df['ema50'] = df['Close'].ewm(span=50, adjust=False).mean()
# --------------------------
# MACD
# --------------------------
df["macd"] = df["ema12"] - df["ema26"]

# MACD Signal
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

# MACD Histogram
df["macd_hist"] = df["macd"] - df["macd_signal"]

In [ ]:
df["ema_slope"] = df["ema13"].diff()
df["macd_slope"] = df["macd_hist"].diff()

df["impulse"] = "blue"

df.loc[
    (df["ema_slope"] > 0) & (df["macd_slope"] > 0),
    "impulse"
] = "green"

df.loc[
    (df["ema_slope"] < 0) & (df["macd_slope"] < 0),
    "impulse"
] = "red"

In [ ]:
df["ret_10d"] = df["Close"].shift(-10) / df["Close"] - 1

In [ ]:
long_cond = (df["Close"] > df["ma20"]) & (df["impulse"] == "green")

long_returns = df.loc[long_cond, "ret_10d"]

In [ ]:
short_cond = (df["Close"] < df["ma20"]) & (df["impulse"] == "red")

short_returns = -df.loc[short_cond, "ret_10d"]

In [ ]:
summary = pd.DataFrame({

    "count":[len(long_returns),len(short_returns)],

    "mean_return":[
        long_returns.mean(),
        short_returns.mean()
    ],

    "median_return":[
        long_returns.median(),
        short_returns.median()
    ],

    "win_rate":[
        (long_returns>0).mean(),
        (short_returns>0).mean()
    ]

}, index=["Long","Short"])

print(summary)

       count  mean_return  median_return  win_rate
Long     446     0.003294       0.005120  0.569507
Short    290    -0.008167      -0.007507  0.396552


#BACKTEST
Close > MA20.
Impulse = Green
Entry on EOD

Exit on EOD
Close Below 20 EMA

In [ ]:
df["entry_signal"] = (
    (df["Close"] > df["ma20"]) &
    (df["impulse"] == "green")
)

In [ ]:
#df["exit_signal"] = df["Close"] < df["ma20"]

In [ ]:
#df["exit_signal"] = df["Close"] < df["ema13"]

In [ ]:
df["exit_signal"] = df["Close"] < df["ema50"]

In [ ]:
position = 0
entry_price = 0

trades = []

for i in range(len(df)):

    price = df["Close"].iloc[i]

    if position == 0:

        if df["entry_signal"].iloc[i]:
            position = 1
            entry_price = price
            entry_date = df.index[i]

    elif position == 1:

        if df["exit_signal"].iloc[i]:

            exit_price = price
            exit_date = df.index[i]

            ret = (exit_price / entry_price) - 1

            trades.append({
                "entry_date": entry_date,
                "exit_date": exit_date,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "return": ret
            })

            position = 0

In [ ]:
trades_df = pd.DataFrame(trades)

In [ ]:
stats = {
    "trades": len(trades_df),
    "mean_return": trades_df["return"].mean(),
    "median_return": trades_df["return"].median(),
    "win_rate": (trades_df["return"] > 0).mean(),
    "max_gain": trades_df["return"].max(),
    "max_loss": trades_df["return"].min(),
}

print(stats)

{'trades': 55, 'mean_return': np.float64(0.012201378056479865), 'median_return': -0.0011312077087264338, 'win_rate': np.float64(0.43636363636363634), 'max_gain': 0.18280384113875914, 'max_loss': -0.035991674020383946}
